# Kotoba Japanese STT Streaming - Real-time Japanese Speech-to-Text on SageMaker

This notebook deploys the **Kotoba Japanese Speech-to-Text (STT) Streaming** model package from AWS Marketplace and transcribes audio over a SageMaker bidirectional stream, end to end.

**Product**: Kotoba Japanese Speech-to-Text (STT) Streaming
**Seller**: Kotoba Technologies

## Key Features

- Real-time Japanese transcription while audio is still arriving
- Katakana person names and clean transcripts without fillers (`style_preference`)
- Keyword biasing for up to 1,000 terms
- Optional turn detection (`server_vad` or sentence-aware `semantic_vad`)
- Optional timestamps: where each delta and turn falls in the input audio
- `pcm16`, `float32`, `mulaw` / `twilio` and Ogg/Opus input at any sample rate
- Billed per hour of input audio

The full protocol reference is in [README.md](./data/README.md).

---

## IMPORTANT: Bidirectional streaming only

The endpoint is **not** a standard REST endpoint:

- `POST /invocations` does **not** transcribe audio
- Batch transform jobs are **not** supported
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]==0.13.1` and `smithy-http[awscrt]` (this notebook uses the 0.13 client API)

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for the HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba Japanese STT Streaming

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateModel",
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DeleteModel",
                "sagemaker:DescribeEndpoint",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithBidirectionalStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Inference](#4.-Perform-Real-time-Inference)
5. [Troubleshooting](#5.-Troubleshooting)
6. [Clean-up](#6.-Clean-up)

## 1. Subscribe to Model Package

1. Open the [Kotoba Japanese Speech-to-Text (STT) Streaming listing](https://aws.amazon.com/marketplace/pp/prodview-vo2nehk65vyn4) on AWS Marketplace
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your region and note the **Model Package ARN**

## 2. Setup

### 2.1 Install Dependencies

Run this cell first. The Python version check in 2.2 does not need these packages, but every cell after it does.

In [ ]:
!pip install -q --upgrade boto3 numpy soundfile
!pip install -q --upgrade "sagemaker>=2.0,<3"
# The HTTP/2 SDK is not yet 1.0 and changes its API between minor versions: this notebook
# uses 0.13. smithy-http[awscrt] adds the HTTP/2 transport that bidirectional streaming needs.
!pip install -q --upgrade "aws-sdk-python[sagemaker-runtime-http2]==0.13.1" "smithy-http[awscrt]"

### 2.2 Check Python Version

`aws-sdk-python` requires Python 3.12+. If this cell raises, switch your kernel (or create a new environment) before continuing.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")

### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
from pathlib import Path

import boto3
import numpy as np
import sagemaker
import soundfile as sf

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.aio.client import AsyncSageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.aio.config import AsyncSageMakerRuntimeHTTP2Config
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from sagemaker import ModelPackage
from smithy_http.aio.crt import AWSCRTHTTPClient

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

SageMaker needs an **execution role** (an IAM *role*, not an IAM *user*) to assume when creating the endpoint.

- **On SageMaker Studio / Notebook Instance**: leave `SAGEMAKER_ROLE_ARN = None`. `sagemaker.get_execution_role()` picks up the attached role.
- **Running locally (IAM user / CLI profile)**: set `SAGEMAKER_ROLE_ARN` to the ARN of a role whose trust policy allows `sagemaker.amazonaws.com` to assume it and which has (at minimum) the `AmazonSageMakerFullAccess` managed policy.

In [ ]:
# --- Execution role -----------------------------------------------------
# If running on SageMaker Studio or a Notebook Instance, leave this as None.
# If running locally as an IAM user, set it to the ARN of a SageMaker-capable role, e.g.
#   SAGEMAKER_ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
SAGEMAKER_ROLE_ARN: str | None = None
# ------------------------------------------------------------------------

sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

if SAGEMAKER_ROLE_ARN is not None:
    role = SAGEMAKER_ROLE_ARN
else:
    try:
        role = sagemaker.get_execution_role()
    except ValueError as exc:
        raise ValueError(
            "get_execution_role() only works in SageMaker-managed environments. "
            "Set SAGEMAKER_ROLE_ARN above to a SageMaker execution role ARN "
            "(not your IAM user ARN). See the markdown cell above for role requirements."
        ) from exc

print(f"Region: {region}")
print(f"Role:   {role}")

### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription. It must be the ARN for the region printed above.

In [ ]:
# TODO: Replace with your Model Package ARN from AWS Marketplace
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )

## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

The model runs on one GPU. Supported instance types:

| Instance types | GPU | Concurrent sessions per instance |
|----------------|-----|----------------------------------|
| `ml.g6.xlarge`, `ml.g6.2xlarge`, `ml.g6.4xlarge`, `ml.g6.8xlarge`, `ml.g6.16xlarge` | 1x NVIDIA L4 | up to 48 |
| `ml.g6e.xlarge`, `ml.g6e.2xlarge`, `ml.g6e.4xlarge`, `ml.g6e.8xlarge`, `ml.g6e.16xlarge` | 1x NVIDIA L40S | up to 128 |

A connection over capacity is refused before the stream opens; raise the instance count for more sessions.

In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-japanese-stt-{int(time.time())}"
INSTANCE_TYPE = "ml.g6e.2xlarge"  # any instance type listed above

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")

In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint (this takes several minutes)
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is InService.")

## 4. Perform Real-time Inference

### 4.1 Connection Protocol

The endpoint is reached through SageMaker's bidirectional-streaming transport:

```
Client (HTTP/2 SDK) -> SageMaker Runtime (HTTPS:8443) -> Container
```

Standard WebSocket libraries **cannot** connect to SageMaker endpoints directly; use `aws-sdk-python[sagemaker-runtime-http2]` with model invocation path `invocations-bidirectional-stream`.

### 4.2 Protocol Flow

```
Client                                         Server
  |<----- transcription_session.created -------|
  |------ transcription_session.update ------->|   (once)
  |<----- transcription_session.updated -------|
  |------ input_audio_buffer.append ---------->|   (repeated, streaming)
  |<----- conversation.item.created -----------|   (once, on the first append)
  |<----- conversation.item.input_audio_transcription.delta --------|   (repeated)
  |<----- conversation.item.input_audio_transcription.completed ----|   (per turn, with turn detection)
  |------ input_audio_buffer.commit ---------->|
  |<----- input_audio_buffer.committed --------|   (then the server closes the stream)
```

### 4.3 Session Timeouts

| Phase | Server behavior |
|-------|-----------------|
| Session init | `transcription_session.update` must be accepted within **60 s** of connecting, or the server sends a `timeout_error` and closes the stream. |
| Established session | No server-side inactivity timeout. Enforce your own client-side read timeout. |
| After commit | The server sends `input_audio_buffer.committed` once the remaining audio is transcribed, then closes the stream. |

See [README.md](./data/README.md) for every session field, event and error code, and [data/sample_input.json](./data/sample_input.json) / [data/sample_output.json](./data/sample_output.json) for event payloads.

### 4.4 Prepare Input Audio

Two paths:

- **Smoke test** (next cell): 3 seconds of silence, to confirm the endpoint is reachable and the protocol round-trips. The transcript will be empty.
- **Real audio** (cell after): loads an audio file for an actual transcription.

In [ ]:
# --- Connectivity smoke test (silent audio) ---------------------------------
# 3 s of silence through the full transcription_session -> commit flow.
# Expect an EMPTY transcript; the goal is to confirm the endpoint routes events.

SAMPLE_RATE = 24000
CHUNK_DURATION_MS = 80  # recommended chunk size

smoke_test_samples = np.zeros(SAMPLE_RATE * 3, dtype=np.int16)
smoke_test_bytes = smoke_test_samples.tobytes()

print(f"Smoke-test audio ready: {len(smoke_test_bytes)} bytes "
      f"({len(smoke_test_samples) / SAMPLE_RATE:.1f} s @ {SAMPLE_RATE} Hz, pcm16)")

### 4.4b Load Your Own Audio (recommended)

Set `AUDIO_PATH` to any Japanese speech file readable by [`soundfile`](https://pysoundfile.readthedocs.io/) (WAV / FLAC / OGG / ...). The server resamples any rate and downmixes PCM channels, so the file is sent as-is in `pcm16`.

In [ ]:
# ----- Replace THIS with your own audio file -------------------------------
AUDIO_PATH = "path/to/audio.wav"   # any format supported by libsndfile (WAV, FLAC, OGG, ...)
# ---------------------------------------------------------------------------


def load_pcm16(path: str) -> tuple[bytes, int, int]:
    """Return (interleaved int16 little-endian PCM bytes, sample_rate, channels)."""
    data, samplerate = sf.read(path, dtype="int16", always_2d=True)
    return data.tobytes(), samplerate, data.shape[1]


if Path(AUDIO_PATH).exists():
    real_audio_bytes, real_audio_rate, real_audio_channels = load_pcm16(AUDIO_PATH)
    duration_s = len(real_audio_bytes) / 2 / real_audio_channels / real_audio_rate
    print(
        f"Loaded {AUDIO_PATH}: {duration_s:.2f} s @ {real_audio_rate} Hz, "
        f"{real_audio_channels} channel(s)"
    )
else:
    real_audio_bytes = real_audio_rate = real_audio_channels = None
    print(
        f"AUDIO_PATH={AUDIO_PATH!r} not found. Update the path above or skip this cell "
        "and run the smoke test only."
    )

### 4.5 Streaming Inference

`transcribe()` opens the HTTP/2 stream, waits for `transcription_session.created`, sends the session settings, streams audio in 80 ms chunks at real-time pace and collects the transcript until `input_audio_buffer.committed`.

Main session options (full table in [README.md](./data/README.md#session-configuration)):

| Field | Default | Notes |
|-------|---------|-------|
| `input_audio_transcription.target_language` | *(required)* | `"ja"` |
| `input_audio_transcription.keywords` | `None` | Up to 1,000 terms of up to 64 characters. |
| `input_audio_transcription.style_preference.human_name` | `"default"` | `"kana"` writes person names in katakana. |
| `input_audio_transcription.style_preference.verbatim` | `True` | `False` removes fillers and restarts (えーと, あの). |
| `input_audio_transcription.timestamps` | `False` | `True` adds `start_ms` / `end_ms` (input-audio milliseconds) to deltas and turns. |
| `turn_detection` | `False` | `{"type": "server_vad", "silence_duration_ms": 800}` or `{"type": "semantic_vad", "eagerness": "medium"}` to receive one `completed` event per turn. |

With turn detection, the transcript of each turn comes from its `completed` event; without it, the transcript is the concatenation of the `delta` events.

In [ ]:
SESSION_INIT_TIMEOUT = 60.0     # the server's init window for transcription_session.update
SESSION_RESULT_TIMEOUT = 60.0   # client-side read timeout while waiting for results


async def transcribe(
    endpoint_name: str,
    audio_bytes: bytes,
    *,
    sample_rate: int = SAMPLE_RATE,
    channels: int = 1,
    keywords: list[str] | None = None,
    human_name: str = "default",
    verbatim: bool = True,
    timestamps: bool = False,
    turn_detection: dict | bool = False,
) -> str:
    """Transcribe pcm16 audio via Kotoba Japanese STT Streaming and return the full transcript."""

    credentials = boto3.Session().get_credentials().get_frozen_credentials()
    config = await AsyncSageMakerRuntimeHTTP2Config.resolve(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        transport=AWSCRTHTTPClient(),  # HTTP/2, needed for bidirectional streaming
    )
    client = AsyncSageMakerRuntimeHTTP2Client(config=config)

    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )

    # data_type="UTF8" makes SageMaker deliver the payload as a text frame;
    # without it the server receives a binary frame and cannot read the event.
    async def send_event(payload: dict) -> None:
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        await stream.input_stream.send(
            RequestStreamEventPayloadPart(
                value=RequestPayloadPart(bytes_=body, data_type="UTF8")
            )
        )

    output = await stream.await_output()
    output_stream = output[1] if isinstance(output, tuple) else output

    async def next_event() -> dict:
        message = await output_stream.receive()
        if message is None:
            raise RuntimeError("Stream closed unexpectedly")
        return json.loads(message.value.bytes_.decode("utf-8"))

    # 1) Wait for transcription_session.created, configure, wait for updated
    async with asyncio.timeout(SESSION_INIT_TIMEOUT):
        event = await next_event()
        if event["type"] == "error":
            raise RuntimeError(f"Server error during init: {event}")
        assert event["type"] == "transcription_session.created", event
        print("[session] created")

        await send_event(
            {
                "type": "transcription_session.update",
                "session": {
                    "input_audio_format": "pcm16",
                    "input_audio_sample_rate": sample_rate,
                    "input_audio_number_of_channels": channels,
                    "input_audio_transcription": {
                        "language": "ja",
                        "target_language": "ja",
                        "keywords": keywords,
                        "style_preference": {"human_name": human_name, "verbatim": verbatim},
                        "timestamps": timestamps,
                    },
                    "turn_detection": turn_detection,
                },
            }
        )

        event = await next_event()
        if event["type"] == "error":
            raise RuntimeError(f"Server error during update: {event}")
        assert event["type"] == "transcription_session.updated", event
        print(f"[session] updated: turn_detection={event['session'].get('turn_detection')}")

    # 2) Stream audio chunks (sender) while collecting results (receiver)
    delta_parts: list[str] = []
    turns: list[str] = []
    chunk_bytes = int(sample_rate * CHUNK_DURATION_MS / 1000) * 2 * channels  # pcm16 = 2 bytes/sample

    async def send_audio() -> None:
        for i in range(0, len(audio_bytes), chunk_bytes):
            await send_event(
                {
                    "event_id": f"chunk_{i // chunk_bytes:07d}",
                    "type": "input_audio_buffer.append",
                    "audio": base64.b64encode(audio_bytes[i : i + chunk_bytes]).decode(),
                }
            )
            await asyncio.sleep(CHUNK_DURATION_MS / 1000)  # real-time pacing
        await send_event({"type": "input_audio_buffer.commit", "event_id": "commit"})

    async def receive_results() -> None:
        while True:
            async with asyncio.timeout(SESSION_RESULT_TIMEOUT):
                event = await next_event()
            etype = event["type"]
            if etype == "conversation.item.input_audio_transcription.delta":
                delta_parts.append(event["delta"])
                print(event["delta"], end="", flush=True)
            elif etype == "conversation.item.input_audio_transcription.completed":
                turns.append(event["transcript"])
                # start_ms / end_ms are absent without timestamps, or if the server lost the alignment.
                span = f" {event['start_ms']}-{event['end_ms']} ms" if "start_ms" in event else ""
                print(f"\n[turn{span}] {event['transcript']}")
            elif etype == "input_audio_buffer.committed":
                print(f"\n[committed] audio_processed_ms={event['audio_processed_ms']}")
                return
            elif etype == "error":
                raise RuntimeError(f"Server error: {event}")

    # A TaskGroup cancels the sender when the receiver fails, unlike gather().
    try:
        async with asyncio.TaskGroup() as tasks:
            tasks.create_task(send_audio())
            tasks.create_task(receive_results())
    finally:
        await stream.input_stream.close()

    return "".join(turns) if turn_detection else "".join(delta_parts)

### 4.6 Run Transcription

**Smoke test first**: confirms the endpoint is healthy. Expect an empty transcript.

In [ ]:
smoke_transcript = await transcribe(
    endpoint_name=ENDPOINT_NAME,
    audio_bytes=smoke_test_bytes,
    sample_rate=SAMPLE_RATE,
    verbatim=False,
)
print(f"\nSmoke-test transcript (expected empty): {smoke_transcript!r}")

**Now the real audio**: a clean transcript with katakana person names and keyword biasing.

In [ ]:
if real_audio_bytes is None:
    print("Skipping: real audio not loaded. Set AUDIO_PATH in the loader cell and re-run.")
else:
    transcript = await transcribe(
        endpoint_name=ENDPOINT_NAME,
        audio_bytes=real_audio_bytes,
        sample_rate=real_audio_rate,
        channels=real_audio_channels,
        keywords=["Kotoba"],
        human_name="kana",
        verbatim=False,
    )
    print(f"\n--- Full transcript ---\n{transcript}")

**With turn detection**: `semantic_vad` ends a turn soon after each finished sentence, and the server sends one `completed` event per turn. Tune `eagerness` (`low` / `medium` / `high`) for your application.

In [ ]:
if real_audio_bytes is None:
    print("Skipping: real audio not loaded. Set AUDIO_PATH in the loader cell and re-run.")
else:
    transcript = await transcribe(
        endpoint_name=ENDPOINT_NAME,
        audio_bytes=real_audio_bytes,
        sample_rate=real_audio_rate,
        channels=real_audio_channels,
        verbatim=False,
        turn_detection={"type": "semantic_vad", "eagerness": "medium"},
    )
    print(f"\n--- Full transcript ---\n{transcript}")

**With timestamps**: `timestamps=True` adds `start_ms` / `end_ms` to each delta with text and each turn: where the text falls in the audio you sent, in milliseconds. With turn detection on, each `[turn ...]` line shows its span.

In [ ]:
if real_audio_bytes is None:
    print("Skipping: real audio not loaded. Set AUDIO_PATH in the loader cell and re-run.")
else:
    transcript = await transcribe(
        endpoint_name=ENDPOINT_NAME,
        audio_bytes=real_audio_bytes,
        sample_rate=real_audio_rate,
        channels=real_audio_channels,
        verbatim=False,
        timestamps=True,
        turn_detection={"type": "semantic_vad", "eagerness": "medium"},
    )
    print(f"\n--- Full transcript ---\n{transcript}")

## 5. Troubleshooting

| Symptom | Likely cause & fix |
|---------|--------------------|
| `RuntimeError: Python 3.12+ is required` | Your kernel is < 3.12. Create a new environment with 3.12+ and re-attach the kernel. |
| `ModuleNotFoundError: aws_sdk_sagemaker_runtime_http2` | The HTTP/2 SDK install failed. Re-run cell 2.1, and restart the kernel on SageMaker Studio. |
| `ImportError` from `aws_sdk_sagemaker_runtime_http2` (for example `cannot import name 'AsyncSageMakerRuntimeHTTP2Config'`) | Another version of the HTTP/2 SDK is installed; this notebook needs 0.13. Re-run cell 2.1, which pins it, and restart the kernel. |
| `UnsupportedTransportError`: the transport `AIOHTTPClient` does not support duplex event streaming | The client was built without the HTTP/2 transport. Keep `transport=AWSCRTHTTPClient()` in the config, and re-run cell 2.1 so that `smithy-http[awscrt]` is installed. |
| Error opening the stream, or `TimeoutError` waiting for `transcription_session.created` | The endpoint is not `InService` yet, the endpoint name is wrong, or the region doesn't match the endpoint's. Check with `sagemaker_session.describe_endpoint(EndpointName=ENDPOINT_NAME)`. |
| `Server error during update` with `invalid_parameters` | A session field is invalid; `error.param` names it. Common causes: `target_language` not `"ja"`, more than 1,000 keywords or a keyword over 64 characters. |
| `Server error` with `payload_too_large` | One `append` carried more than 8 seconds of audio. Send smaller chunks (80 ms recommended). |
| The stream is refused with HTTP 503 (`system_busy` or `worker_unavailable`) | The endpoint is at its session capacity, or the model is not ready. Retry with backoff or raise the instance count. |
| `Server error` with `invalid_parameters` and `param` `audio` | An `append` carried audio that cannot be decoded, for example PCM whose length is not a whole number of samples × channels. Check `input_audio_format`, `input_audio_sample_rate` and `input_audio_number_of_channels` against the bytes you send. |
| `Server error` with `server_error`, then the stream closes | An internal error ended the session. Start a new session. |
| Transcript contains fillers (えーと, あの) | `style_preference.verbatim` defaults to `true` (as does `transcribe()`). Pass `verbatim=False` for a clean transcript. |
| Transcript is empty on real audio | Confirm the audio is Japanese speech and audible, and that `input_audio_sample_rate` / `input_audio_number_of_channels` match the bytes you send. |
| `AccessDeniedException` at deploy time | Your IAM role lacks the permissions listed under "Required IAM Permissions", or the AWS Marketplace subscription is not complete. |
| Connection closes ~60 s after connect, before `transcription_session.updated` | `transcription_session.update` was not accepted within the server's 60 s init window. Send it right after `transcription_session.created`. |
| Session opens but sits idle until your client read-timeout fires | Expected when no audio is sent: the server has no inactivity timeout. Close idle sessions from the client. |

## 6. Clean-up

**Important**: Delete the endpoint when done. SageMaker bills the GPU instance **per hour while the endpoint exists**, even when idle.

In [ ]:
print(f"Deleting endpoint '{ENDPOINT_NAME}'...")

predictor.delete_endpoint(delete_endpoint_config=True)
predictor.delete_model()

print("Endpoint deleted.")

### Unsubscribe from Model Package (Optional)

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba Japanese Speech-to-Text (STT) Streaming" in your subscriptions
3. Click **Manage** -> **Actions** -> **Cancel subscription**

**Note**: Delete all endpoints using this model package before unsubscribing.

---

## Support

For technical support, contact kotoba_product+marketplace@kotoba.tech.

## References

- Protocol reference: [README.md](./data/README.md)
- Event samples: [data/sample_input.json](./data/sample_input.json), [data/sample_output.json](./data/sample_output.json)
- [SageMaker real-time endpoints](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)